# End-to-End Hybrid RAG Pipeline: Ingestion, Chunking & Deduplication (Phase 1)

In [2]:
import os
import re
import shutil
import numpy as np
from html.parser import HTMLParser
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Hybrid and Reranking Imports
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

C:\Users\Ayush Mohanty\AppData\Local\Temp\ipykernel_13100\3224611080.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader
d:\programs\Mini-Project\rag-streamlit-app\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Multi-Format Document Loader

In [3]:
class HTMLTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text_parts = []
        
    def handle_data(self, data):
        self.text_parts.append(data)
        
    def get_text(self):
        return "".join(self.text_parts)

file_path = "fepr102.pdf"
print(f"Loading file: {file_path}")

file_extension = file_path.split('.')[-1].lower()
try:
    if file_extension == 'pdf':
        loader = PyPDFLoader(file_path)
        documents = loader.load()
    elif file_extension in ['txt', 'md']:
        loader = TextLoader(file_path, encoding='utf-8')
        documents = loader.load()
    elif file_extension in ['html', 'htm']:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            html_content = f.read()
        parser = HTMLTextExtractor()
        parser.feed(html_content)
        documents = [Document(page_content=parser.get_text(), metadata={"source_filename": file_path})]
        
    # Standardize metadata
    for doc in documents:
        doc.metadata["source_filename"] = file_path
        if "page" not in doc.metadata:
            doc.metadata["page"] = 1
            
    print(f"Successfully loaded {len(documents)} pages.")
except Exception as e:
    print(f"Error loading file: {e}")
    documents = []

Loading file: fepr102.pdf
Successfully loaded 36 pages.


# 2. Configurable Chunking Strategies

In [ ]:
def extract_section_heading(text):
    lines = text.split("\n")
    for line in lines:
        line = line.strip()
        if line.startswith("#"):
            return line.lstrip("#").strip()
        if line.isupper() and len(line) > 3 and len(line) < 80:
            return line
        if re.match(r'^\d+(\.\d+)*\s+[A-Z]', line):
            return line
    return "General Context"

def chunk_documents(documents, strategy, chunk_size=3000, chunk_overlap=600, embeddings_model=None):
    chunks = []
    
    if strategy == "fixed-size":
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        raw_chunks = text_splitter.split_documents(documents)
        for i, rc in enumerate(raw_chunks):
            section = extract_section_heading(rc.page_content)
            rc.metadata.update({
                "chunk_index": i,
                "section_heading": section,
                "chunking_strategy": "fixed-size",
                "character_count": len(rc.page_content)
            })
            chunks.append(rc)
            
    elif strategy == "header-aware":
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n# ", "\n## ", "\n### ", "\n#### ", "\n", " ", ""]
        )
        raw_chunks = text_splitter.split_documents(documents)
        for i, rc in enumerate(raw_chunks):
            section = extract_section_heading(rc.page_content)
            rc.metadata.update({
                "chunk_index": i,
                "section_heading": section,
                "chunking_strategy": "header-aware",
                "character_count": len(rc.page_content)
            })
            chunks.append(rc)
            
    elif strategy == "semantic":
        chunk_idx = 0
        for doc in documents:
            sentences = [s.strip() for s in re.split(r'(?<=[.?!])\s+', doc.page_content) if s.strip()]
            if not sentences:
                continue
                
            try:
                sentence_embeddings = embeddings_model.embed_documents(sentences)
            except Exception as e:
                
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
                raw_chunks = text_splitter.split_documents([doc])
                for rc in raw_chunks:
                    section = extract_section_heading(rc.page_content)
                    rc.metadata.update({
                        "chunk_index": chunk_idx,
                        "section_heading": section,
                        "chunking_strategy": "semantic-fallback",
                        "character_count": len(rc.page_content)
                    })
                    chunks.append(rc)
                    chunk_idx += 1
                continue
            
            current_chunk_sentences = [sentences[0]]
            
            for i in range(len(sentences) - 1):
                vec1 = np.array(sentence_embeddings[i])
                vec2 = np.array(sentence_embeddings[i+1])
                
                norm1 = np.linalg.norm(vec1)
                norm2 = np.linalg.norm(vec2)
                similarity = np.dot(vec1, vec2) / (norm1 * norm2) if norm1 > 0 and norm2 > 0 else 0.0
                
                current_len = sum(len(s) for s in current_chunk_sentences)
                if similarity < 0.75 or current_len + len(sentences[i+1]) > chunk_size:
                    chunk_text = " ".join(current_chunk_sentences)
                    section = extract_section_heading(chunk_text)
                    chunks.append(Document(
                        page_content=chunk_text,
                        metadata={
                            "source_filename": doc.metadata.get("source_filename", "Unknown"),
                            "page": doc.metadata.get("page", 1),
                            "chunk_index": chunk_idx,
                            "section_heading": section,
                            "chunking_strategy": "semantic",
                            "character_count": len(chunk_text)
                        }
                    ))
                    chunk_idx += 1
                    current_chunk_sentences = [sentences[i+1]]
                else:
                    current_chunk_sentences.append(sentences[i+1])
            
            if current_chunk_sentences:
                chunk_text = " ".join(current_chunk_sentences)
                section = extract_section_heading(chunk_text)
                chunks.append(Document(
                    page_content=chunk_text,
                    metadata={
                        "source_filename": doc.metadata.get("source_filename", "Unknown"),
                        "page": doc.metadata.get("page", 1),
                        "chunk_index": chunk_idx,
                        "section_heading": section,
                        "chunking_strategy": "semantic",
                        "character_count": len(chunk_text)
                    }
                ))
                chunk_idx += 1
                
    return chunks

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


chosen_strategy = "semantic"
chunks = chunk_documents(documents, chosen_strategy, chunk_size=3000, chunk_overlap=600, embeddings_model=embeddings_model)
print(f"Created {len(chunks)} chunks using '{chosen_strategy}' strategy.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10615.88it/s]


Created 601 chunks using 'semantic' strategy.


# 3. Deduplication and Embedding Storage

In [6]:
def deduplicate_chunks(chunks, vector_store, embeddings):
    unique_chunks = []
    has_existing = False
    try:
        if vector_store and len(vector_store.get()['ids']) > 0:
            has_existing = True
    except Exception:
        pass
        
    for chunk in chunks:
        is_duplicate = False
        if unique_chunks:
            for uc in unique_chunks:
                if uc.page_content.strip() == chunk.page_content.strip():
                    is_duplicate = True
                    break
                    
        if not is_duplicate and has_existing:
            try:
                results = vector_store.similarity_search_with_score(chunk.page_content, k=1)
                if results:
                    _, score = results[0]
                    if score < 0.05:
                        is_duplicate = True
            except Exception:
                pass
                
        if not is_duplicate:
            unique_chunks.append(chunk)
            
    return unique_chunks

print("Initializing Chroma DB...")
persist_directory = os.path.join(os.getcwd(), "chroma_db_notebook")
vector_store = Chroma(
    collection_name="rag_collection_notebook",
    embedding_function=embeddings_model,
    persist_directory=persist_directory,
    collection_metadata={"hnsw:space": "cosine"}
)

print("Deduplicating...")
unique_chunks = deduplicate_chunks(chunks, vector_store, embeddings_model)
print(f"Kept {len(unique_chunks)} unique chunks out of {len(chunks)}.")

if unique_chunks:
    try:
        vector_store.delete_collection()
    except Exception:
        pass
        
    vector_store = Chroma.from_documents(
        documents=unique_chunks, 
        embedding=embeddings_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    bm25_retriever = BM25Retriever.from_documents(unique_chunks)
    print("Chroma DB and BM25 index populated in sync!")

Initializing Chroma DB...
Deduplicating...
Kept 551 unique chunks out of 601.
Chroma DB and BM25 index populated in sync!


# 4. Hybrid Retrieval and Cross-Encoder Reranking Setup

In [7]:
top_k = 5
initial_k = 20
bm25_weight = 0.3

# 1. Base Dense Retriever
dense_retriever = vector_store.as_retriever(search_kwargs={"k": initial_k})

# 2. Base Sparse BM25 Retriever
bm25_retriever.k = initial_k

# 3. Fusion Layer (RRF via EnsembleRetriever)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[bm25_weight, 1.0 - bm25_weight]
)

# 4. Reranker Stage
print("Loading Cross-Encoder Reranker...")
cross_encoder_model = HuggingFaceCrossEncoder(model_name="ms-marco-MiniLM-L-12-v2")
compressor = CrossEncoderReranker(model=cross_encoder_model, top_n=top_k)
hybrid_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=ensemble_retriever
)
print("Retriever pipeline setup successfully!")

Loading Cross-Encoder Reranker...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6339.62it/s]


Retriever pipeline setup successfully!


# 5. Query the RAG Pipeline

In [8]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt_template)
rag_chain = create_retrieval_chain(hybrid_rerank_retriever, question_answer_chain)
print("RAG Chain compiled!")

RAG Chain compiled!


In [9]:
question = "Who was the friend of Gajaraj?"
print(f"Asking: {question}")

response = rag_chain.invoke({"input": question})
print(f"\nAnswer: {response['answer']}\n")

print("Retrieved Context Sources:")
for i, doc in enumerate(response.get("context", [])):
    print(f"\n--- Source {i+1} ---")
    print(f"File: {doc.metadata.get('source_filename', 'Unknown')}")
    print(f"Heading: {doc.metadata.get('section_heading', 'General Context')}")
    print(f"Strategy: {doc.metadata.get('chunking_strategy', 'Unknown')}")
    print(f"Snippet: {doc.page_content[:150]}...")

Asking: Who was the friend of Gajaraj?

Answer: The friend of Gajaraj was Buntee.

Retrieved Context Sources:

--- Source 1 ---
File: fepr102.pdf
Heading: General Context
Strategy: semantic
Snippet: Gajaraj was sad without a friend and when he met Buntee, he was filled 
with joy....

--- Source 2 ---
File: fepr102.pdf
Heading: General Context
Strategy: semantic
Snippet: “It’s not only Gajaraj who has found a friend,” 
said the mahout hugging the farmer, “I’ve also 
found one.”
subba rao
kathakids.com
satisfaction: 
ha...

--- Source 3 ---
File: fepr102.pdf
Heading: General Context
Strategy: semantic
Snippet: Let us speak
  Gajaraj and Buntee had a wonderful time with each other even if they 
were ‘unlik
ely’ friends....

--- Source 4 ---
File: fepr102.pdf
Heading: General Context
Strategy: semantic
Snippet: V What was ‘unlikely’ about the friendship of Gajaraj and Buntee?...

--- Source 5 ---
File: fepr102.pdf
Heading: General Context
Strategy: semantic
Snippet: “I wish I had a friend 